# ML-CAP-01 — Ranking the content pages most likely to lose search visibility next month

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAyyanHassan/flyrank-ml-internship-work/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring  
**Primary frame:** March 2026 features → April 2026 outcome  
**Forward-time test:** April 2026 features → May 2026 outcome  
**Task type:** ranking / prioritization. **Primary metric:** Precision@K for K = 10, 50, 100, 500, always printed next to the base rate.

This notebook is the computational half of the capstone. It builds the review queue, scores a transparent rule baseline and two models on the *same* client-grouped split, puts bootstrap uncertainty bands on Precision@K, runs a feature ablation to separate momentum from genuine early-warning signal, and then re-tests the model on a month it never saw during fitting.

Every number the deployed paper cites is printed here and written to `work/outputs/capstone_metrics.json` as a run receipt.

> **Public-safe rule:** no client names, domains, URLs, private queries, credentials or raw exports. Observed / measured / directional / decision-support language only. Nothing here explains or predicts Google ranking behavior, and nothing here claims a refresh caused a recovery.

## 1. Question

**Research question.** Using only search behavior that is already observable at decision time, can we rank a content library so that the pages most likely to lose search visibility next month appear at the top of a short review queue — beating a transparent rule baseline on the same split, and holding up on a future month?

**The decision this supports.** A content team can review a limited number of pages per month. The queue answers one operational question: *which pages should a human look at first?* The output is a prioritization aid, not a verdict on any single page.

**Outcome definition (proxy label).** `future_decline_label = 1` when a page had positive impressions in the decision window and its impressions in the following month fell more than 20% below that decision-window total. This is a measurable proxy for losing search visibility. It is not a measure of revenue, of rankings, or of content quality.

**Why a ranking metric instead of accuracy.** The team acts on a top-K slice, so the useful question is what share of the top K later matched the outcome. Precision@K appears next to the base rate in every table, so the reader sees the lift rather than an unanchored percentage.

In [1]:
%pip -q install duckdb pandas numpy scikit-learn matplotlib

import json
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is missing. Add your Read token to Colab Secrets as HF_TOKEN.')

con = duckdb.connect()
con.execute('INSTALL httpfs;')
con.execute('LOAD httpfs;')
con.execute('CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN ?)', [HF_TOKEN])

WAREHOUSE = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
RELEASE_ID = 'flyrank_pseudonymized_warehouse_release_v20260703'

DECISION_MONTH = '2026-03'
OUTCOME_MONTH = '2026-04'
FWD_DECISION_MONTH = '2026-04'
FWD_OUTCOME_MONTH = '2026-05'

RANDOM_STATE = 42
KS = [10, 50, 100, 500]
DECLINE_RATIO = 0.80
BOOTSTRAP_REPS = 2000
VOLUME_THRESHOLD = 500
POSITION_BAND = (4, 20)

KEY_COLS = ['client_hash_id', 'content_hash_id']
GROUP = 'client_hash_id'
TARGET = 'future_decline_label'

CORE_FEATURES = ['w_impressions', 'w_clicks', 'w_ctr_pct', 'w_avg_position', 'w_impression_days']
DYNAMICS_FEATURES = ['w_late_share', 'w_max_day_share', 'w_impr_cv', 'w_active_day_share']
FEATURES = CORE_FEATURES + DYNAMICS_FEATURES

OUTPUT_DIR = Path('work/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Warehouse connection configured.')
print('Release:', RELEASE_ID)
print('Primary frame:', DECISION_MONTH, 'features ->', OUTCOME_MONTH, 'outcome')
print('Forward test:', FWD_DECISION_MONTH, 'features ->', FWD_OUTCOME_MONTH, 'outcome')
print('Feature count:', len(FEATURES))
print('Features:', FEATURES)

Warehouse connection configured.
Release: flyrank_pseudonymized_warehouse_release_v20260703
Primary frame: 2026-03 features -> 2026-04 outcome
Forward test: 2026-04 features -> 2026-05 outcome
Feature count: 9
Features: ['w_impressions', 'w_clicks', 'w_ctr_pct', 'w_avg_position', 'w_impression_days', 'w_late_share', 'w_max_day_share', 'w_impr_cv', 'w_active_day_share']


## 2. Data

**Release.** `flyrank_pseudonymized_warehouse_release_v20260703`, the pseudonymized FlyRank internship warehouse. Client and content identifiers are hashes; no names, domains or URLs exist in the tables this notebook reads.

**Table.** `fact_content_daily_performance` — one row per client, content item and calendar day. It is read directly from the gated Hugging Face release through DuckDB over `hf://`, so no dataset file is ever written into the repository.

**Windows.** Features come from a single decision month; the outcome comes from the month immediately after it. Two frames are built by the identical code path:

| Frame | Features from | Outcome from | Role |
| --- | --- | --- | --- |
| A | March 2026 | April 2026 | model development and grouped validation |
| B | April 2026 | May 2026 | forward-time test, never seen during fitting |

**Exclusions, and why each one is there.**

- Rows where `gsc_data_available` is not true are dropped. A missing measurement is not the same as zero visibility, and keeping them would manufacture fake declines.
- A page must appear in both the decision month and the outcome month. Pages absent from the outcome month are excluded because vanishing from the table can mean tracking changed rather than traffic fell; scoring that as a decline would be a measurement artifact, not a finding.
- The `_sample` table is never touched. It holds the last month of the release, so using it would mean peeking at the future.
- Only the two months each frame needs are pulled, rather than the full 78.8M-row fact table. That keeps the run inside the release rate limits and keeps the decision boundary unambiguous.

**Fields used.** Impressions, clicks, impression-weighted average position and daily coverage — all from the decision window only. Identifiers are used for joining, grouping and audit, never as model features.

In [2]:
def month_relation(month):
    return WAREHOUSE + '/month=' + month + '/*.parquet'


def window_features(month):
    rel = month_relation(month)
    sql = f'''
    WITH daily AS (
        SELECT client_hash_id,
               content_hash_id,
               report_date,
               SUM(gsc_impressions) AS day_impressions,
               SUM(gsc_clicks) AS day_clicks,
               SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0
                        THEN gsc_impressions * gsc_avg_position ELSE 0 END) AS pos_num,
               SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0
                        THEN gsc_impressions ELSE 0 END) AS pos_den
        FROM read_parquet('{rel}')
        WHERE month = '{month}' AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id, report_date
    ),
    bounds AS (
        SELECT MAX(report_date) AS w_end,
               COUNT(DISTINCT report_date) AS w_days
        FROM daily
    )
    SELECT d.client_hash_id,
           d.content_hash_id,
           SUM(d.day_impressions) AS w_impressions,
           SUM(d.day_clicks) AS w_clicks,
           CASE WHEN SUM(d.day_impressions) > 0
                THEN 100.0 * SUM(d.day_clicks) / SUM(d.day_impressions)
                ELSE NULL END AS w_ctr_pct,
           SUM(d.pos_num) / NULLIF(SUM(d.pos_den), 0) AS w_avg_position,
           COUNT(DISTINCT CASE WHEN d.day_impressions > 0 THEN d.report_date END) AS w_impression_days,
           SUM(CASE WHEN d.report_date > b.w_end - INTERVAL 7 DAY
                    THEN d.day_impressions ELSE 0 END) AS w_last7_impressions,
           MAX(d.day_impressions) AS w_max_day_impressions,
           STDDEV_SAMP(d.day_impressions) AS w_day_sd,
           AVG(d.day_impressions) AS w_day_mean,
           MAX(b.w_days) AS w_window_days
    FROM daily AS d
    CROSS JOIN bounds AS b
    GROUP BY d.client_hash_id, d.content_hash_id
    '''
    frame = con.execute(sql).df()
    assert not frame.duplicated(KEY_COLS).any(), 'Duplicate key in the decision-window features.'
    return frame


def window_outcome(month):
    rel = month_relation(month)
    sql = f'''
    SELECT client_hash_id,
           content_hash_id,
           SUM(gsc_impressions) AS outcome_impressions
    FROM read_parquet('{rel}')
    WHERE month = '{month}' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    '''
    frame = con.execute(sql).df()
    assert not frame.duplicated(KEY_COLS).any(), 'Duplicate key in the outcome window.'
    return frame


def build_frame(decision_month, outcome_month):
    assert decision_month < outcome_month, 'The outcome window must come after the decision window.'
    feats = window_features(decision_month)
    outcome = window_outcome(outcome_month)
    frame = feats.merge(outcome, on=KEY_COLS, how='inner', validate='one_to_one')
    frame = frame.sort_values(KEY_COLS, kind='mergesort').reset_index(drop=True)
    volume = frame['w_impressions'].replace(0, np.nan)
    frame['w_late_share'] = (frame['w_last7_impressions'] / volume).fillna(0.0)
    frame['w_max_day_share'] = (frame['w_max_day_impressions'] / volume).fillna(0.0)
    frame['w_impr_cv'] = (frame['w_day_sd'] / frame['w_day_mean'].replace(0, np.nan)).fillna(0.0)
    frame['w_active_day_share'] = frame['w_impression_days'] / frame['w_window_days']
    frame[TARGET] = ((frame['w_impressions'] > 0) & (frame['outcome_impressions'] < DECLINE_RATIO * frame['w_impressions'])).astype(int)
    frame['decision_window'] = decision_month
    frame['outcome_window'] = outcome_month
    return frame


outcome_like = {'outcome_impressions', 'trend_direction', 'trend_pct', 'label', 'target',
                'health_score', 'priority_score', 'action_type', TARGET}
assert not (set(FEATURES) & outcome_like), 'A feature overlaps an outcome or label-derived field.'
assert not (set(FEATURES) & set(KEY_COLS)), 'An identifier is being used as a model feature.'
assert all(name.startswith('w_') for name in FEATURES), 'Every feature must come from the decision window.'
print('Decision-time legality asserted: decision-window features only, identifiers excluded.')

frame_a = build_frame(DECISION_MONTH, OUTCOME_MONTH)
frame_b = build_frame(FWD_DECISION_MONTH, FWD_OUTCOME_MONTH)


def frame_summary(frame, name):
    return {'frame': name,
            'features_from': frame['decision_window'].iloc[0],
            'outcome_from': frame['outcome_window'].iloc[0],
            'rows': int(len(frame)),
            'clients': int(frame[GROUP].nunique()),
            'decline_rate': round(float(frame[TARGET].mean()), 4)}


frames_summary = pd.DataFrame([frame_summary(frame_a, 'A primary'), frame_summary(frame_b, 'B forward test')])
display(frames_summary)

REF_ROWS_A = 158549
REF_RATE_A = 0.4782
rows_ok = len(frame_a) == REF_ROWS_A
rate_ok = abs(float(frame_a[TARGET].mean()) - REF_RATE_A) < 0.0005
print('Reproducibility check of Frame A against ML-04 / ML-07 / ML-09:')
print('  rows == 158,549     ->', 'PASS' if rows_ok else 'DIFFERS: ' + format(len(frame_a), ','))
print('  decline rate 0.4782 ->', 'PASS' if rate_ok else 'DIFFERS: ' + format(float(frame_a[TARGET].mean()), '.4f'))
display(frame_a[FEATURES + [TARGET]].describe().T.round(4))

Decision-time legality asserted: decision-window features only, identifiers excluded.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,frame,features_from,outcome_from,rows,clients,decline_rate
0,A primary,2026-03,2026-04,158549,46,0.4782
1,B forward test,2026-04,2026-05,183345,51,0.4721


Reproducibility check of Frame A against ML-04 / ML-07 / ML-09:
  rows == 158,549     -> PASS
  decline rate 0.4782 -> PASS


,count,mean,std,min,25%,50%,75%,max
w_impressions,158549.0,1768.0347,5706.7568,1.0000,40.0000,246.0000,1264.0000,617124.0000
w_clicks,158549.0,5.1767,28.1657,0.0000,0.0000,0.0000,2.0000,5668.0000
w_ctr_pct,158549.0,0.3719,2.5448,0.0000,0.0000,0.0000,0.2514,100.0000
w_avg_position,157790.0,16.9097,18.1783,0.0196,5.2500,8.8622,22.1564,309.0000
w_impression_days,158549.0,22.3052,10.4806,1.0000,14.0000,29.0000,31.0000,31.0000
w_late_share,158549.0,0.2945,0.2360,0.0000,0.1600,0.2361,0.3406,1.0000
w_max_day_share,158549.0,0.1833,0.2077,0.0371,0.0701,0.1026,0.1923,1.0000
w_impr_cv,158549.0,0.5498,0.3238,0.0000,0.3689,0.5193,0.6917,5.3329
w_active_day_share,158549.0,0.7195,0.3381,0.0323,0.4516,0.9355,1.0000,1.0000
future_decline_label,158549.0,0.4782,0.4995,0.0000,0.0000,0.0000,1.0000,1.0000


## 3. Methodology

### Assumptions, stated before any result

1. A drop of more than 20% in next-month impressions is decision-worthy for this lane. The threshold is a business convention, not an estimate.
2. Behavior inside the decision window carries information about the next month. That is an association, not a mechanism.
3. Clients differ from one another, so the honest question is whether the ranking transfers to a client the model has never trained on.

### Features — nine, all from the decision window

**Level features:** `w_impressions`, `w_clicks`, `w_ctr_pct`, `w_avg_position` (impression-weighted), `w_impression_days`.

**Within-window dynamics features:** `w_late_share` (share of window impressions landing in the final seven days), `w_max_day_share` (largest single day as a share of the window), `w_impr_cv` (day-to-day volatility of impressions), `w_active_day_share` (days with impressions divided by days in the window).

Every feature carries a `w_` prefix and the code asserts it. A column whose name does not start with `w_` cannot enter the feature matrix, so an outcome-window field cannot reach the model by accident.

A feature ablation below runs the same logistic regression on the five level features alone and then on all nine, through the identical grouped split, because the dynamics features are partly mechanical with the label and their contribution has to be shown, not hidden.

**Banned by construction:** the outcome-window total, the label itself, `trend_direction`, `trend_pct`, any rebuilt product flag (`health_score`, `priority_score`, `action_type`), and both hash identifiers.

### Baseline — the frozen rule from ML-07

`score = 5` when a page has at least 500 decision-window impressions **and** sits in positions 4 to 20 **and** its CTR is below the median CTR of that same visible band; `score = 3` for volume alone; `score = 0` otherwise. The band median is recomputed inside each training fold and never on the evaluation rows, so the rule and the models are judged on exactly the same information.

### Models

- **Logistic regression** — median imputation, standardization, `max_iter=1000`. The interpretable reference carried forward from ML-08.
- **Histogram gradient boosting** — the complexity test. If it does not clear logistic regression by more than the uncertainty band, that is a reportable negative result, not a failure.

### Validation design

**Grouped cross-validation.** `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` grouped on `client_hash_id`. Every row receives an out-of-fold score from a model that never saw its client, and zero client overlap is asserted fold by fold. ML-09 showed why this matters: the earlier random row split left 42 of 46 clients sitting on both sides of the split.

**Forward-time test.** Fit on Frame A, predict Frame B. Nothing from the later window touches fitting, thresholds or feature construction.

**Uncertainty.** Precision@10 is a mean over ten outcomes, so a single percentage would overstate what the data can support. A bootstrap band is reported for every K. The band describes outcome variability inside the selected top-K set with the ranking held fixed; it is not a confidence interval over redrawing the whole population.

### Reproducibility fix carried into this notebook

The earlier notebooks split on whatever row order DuckDB happened to return, and a `GROUP BY` does not guarantee one. The same seed therefore produced slightly different splits between runs, which is why ML-08 and ML-09 report different AUC for what should have been the same split. Every frame here is sorted on its join keys before anything is split, which makes the run deterministic.

## 4. Results (vs baseline)

Three scorers are compared on one identical client-grouped split: the frozen rule baseline, logistic regression and histogram gradient boosting. Each row is scored out-of-fold by a model that never trained on its client, so the comparison is like for like. Precision@K sits next to the base rate, and a bootstrap band shows how firm each number is — Precision@10 is a mean over only ten outcomes, so its band is wide on purpose.

In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(RANDOM_STATE)
TIEBREAK = ['w_impressions', 'w_clicks'] + KEY_COLS


def make_logreg():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])


def make_histgb():
    return HistGradientBoostingClassifier(random_state=RANDOM_STATE)


def rule_cutoff(train_frame):
    band = train_frame[(train_frame['w_impressions'] >= VOLUME_THRESHOLD)
                       & train_frame['w_avg_position'].between(POSITION_BAND[0], POSITION_BAND[1], inclusive='both')
                       & train_frame['w_ctr_pct'].notna()]
    cutoff = band['w_ctr_pct'].median()
    return float(cutoff) if pd.notna(cutoff) else float('inf')


def rule_score(rows, cutoff):
    high_volume = rows['w_impressions'] >= VOLUME_THRESHOLD
    ctr_fix = (high_volume
               & rows['w_avg_position'].between(POSITION_BAND[0], POSITION_BAND[1], inclusive='both')
               & rows['w_ctr_pct'].notna()
               & rows['w_ctr_pct'].lt(cutoff))
    return np.select([high_volume & ctr_fix, high_volume], [5.0, 3.0], default=0.0)


def top_k_labels(frame, score_col, k):
    k = min(k, len(frame))
    ranked = frame.sort_values([score_col] + TIEBREAK,
                              ascending=[False] + [False] * len(TIEBREAK),
                              kind='mergesort')
    return ranked.head(k)[TARGET].to_numpy()


def bootstrap_ci(labels, reps=BOOTSTRAP_REPS):
    if len(labels) == 0:
        return (float('nan'), float('nan'))
    draws = rng.choice(labels, size=(reps, len(labels)), replace=True).mean(axis=1)
    return (float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5)))


def precision_table(frame, scorers, base_rate):
    rows = []
    for label, col in scorers:
        for k in KS:
            labels = top_k_labels(frame, col, k)
            point = float(labels.mean())
            low, high = bootstrap_ci(labels)
            rows.append({'scorer': label, 'k': k,
                         'precision_pct': round(100 * point, 2),
                         'ci95_low_pct': round(100 * low, 2),
                         'ci95_high_pct': round(100 * high, 2),
                         'base_rate_pct': round(100 * base_rate, 2),
                         'lift_pp': round(100 * (point - base_rate), 2)})
    return pd.DataFrame(rows)


def ranking_table(frame, scorers):
    rows = []
    for label, col in scorers:
        rows.append({'scorer': label,
                     'roc_auc': round(float(roc_auc_score(frame[TARGET], frame[col])), 4),
                     'average_precision': round(float(average_precision_score(frame[TARGET], frame[col])), 4)})
    return pd.DataFrame(rows)


def grouped_oof_model(frame, model_factory, feature_cols=FEATURES):
    scores = np.full(len(frame), np.nan)
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr, va) in enumerate(cv.split(frame[feature_cols], frame[TARGET], groups=frame[GROUP]), start=1):
        assert not (set(frame.iloc[tr][GROUP]) & set(frame.iloc[va][GROUP])), 'Client overlap in fold ' + str(fold)
        model = model_factory().fit(frame.iloc[tr][feature_cols], frame.iloc[tr][TARGET])
        scores[va] = model.predict_proba(frame.iloc[va][feature_cols])[:, 1]
    assert not np.isnan(scores).any(), 'Some rows never received an out-of-fold score.'
    return scores


def grouped_oof_rule(frame):
    scores = np.full(len(frame), np.nan)
    cutoffs = []
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    for tr, va in cv.split(frame[FEATURES], frame[TARGET], groups=frame[GROUP]):
        cutoff = rule_cutoff(frame.iloc[tr])
        cutoffs.append(cutoff)
        scores[va] = rule_score(frame.iloc[va], cutoff)
    assert not np.isnan(scores).any(), 'Some rows never received an out-of-fold rule score.'
    return scores, cutoffs


print('Helpers ready: frozen rule baseline, logistic regression, histogram gradient boosting,')
print('precision@K with bootstrap bands, and grouped out-of-fold scoring for all three.')

Helpers ready: frozen rule baseline, logistic regression, histogram gradient boosting,
precision@K with bootstrap bands, and grouped out-of-fold scoring for all three.


In [4]:
base_rate_a = float(frame_a[TARGET].mean())

frame_a['score_baseline'], fold_cutoffs = grouped_oof_rule(frame_a)
frame_a['score_logreg'] = grouped_oof_model(frame_a, make_logreg)
frame_a['score_histgb'] = grouped_oof_model(frame_a, make_histgb)

SCORERS = [('Rule baseline', 'score_baseline'),
           ('Logistic regression', 'score_logreg'),
           ('Histogram GB', 'score_histgb')]

primary_pk = precision_table(frame_a, SCORERS, base_rate_a)
primary_rank = ranking_table(frame_a, SCORERS)

print('Primary frame: ' + DECISION_MONTH + ' features -> ' + OUTCOME_MONTH + ' outcome')
print('Client-grouped out-of-fold scoring, same split for all three scorers.')
print('Base rate: ' + format(100 * base_rate_a, '.2f') + '%   Rows: ' + format(len(frame_a), ',') + '   Clients: ' + str(frame_a[GROUP].nunique()))
print('Per-fold rule CTR cutoffs (%): ' + ', '.join(format(c, '.4f') for c in fold_cutoffs))
display(primary_pk)
display(primary_rank)
print('AUC and average precision rank the whole population; Precision@K is the operational metric')
print('because the review team acts on a top-K slice, not on the full library.')

pivot_pk = primary_pk.pivot(index='k', columns='scorer', values='precision_pct')
print('Precision@K side by side (%), base rate ' + format(100 * base_rate_a, '.2f') + '%:')
display(pivot_pk)

Primary frame: 2026-03 features -> 2026-04 outcome
Client-grouped out-of-fold scoring, same split for all three scorers.
Base rate: 47.82%   Rows: 158,549   Clients: 46
Per-fold rule CTR cutoffs (%): 0.1911, 0.2155, 0.1907, 0.1886, 0.1734


,scorer,k,precision_pct,ci95_low_pct,ci95_high_pct,base_rate_pct,lift_pp
0,Rule baseline,10,50.0,20.0,80.0,47.82,2.18
1,Rule baseline,50,46.0,32.0,60.0,47.82,-1.82
2,Rule baseline,100,43.0,33.0,53.0,47.82,-4.82
3,Rule baseline,500,49.6,45.2,54.0,47.82,1.78
4,Logistic regression,10,100.0,100.0,100.0,47.82,52.18
5,Logistic regression,50,96.0,90.0,100.0,47.82,48.18
6,Logistic regression,100,93.0,87.0,98.0,47.82,45.18
7,Logistic regression,500,86.8,83.8,89.6,47.82,38.98
8,Histogram GB,10,90.0,70.0,100.0,47.82,42.18
9,Histogram GB,50,90.0,80.0,98.0,47.82,42.18


,scorer,roc_auc,average_precision
0,Rule baseline,0.5249,0.4994
1,Logistic regression,0.6644,0.6378
2,Histogram GB,0.7155,0.6874


AUC and average precision rank the whole population; Precision@K is the operational metric
because the review team acts on a top-K slice, not on the full library.
Precision@K side by side (%), base rate 47.82%:


scorer,Histogram GB,Logistic regression,Rule baseline
k,,,
10,90.0,100.0,50.0
50,90.0,96.0,46.0
100,88.0,93.0,43.0
500,93.4,86.8,49.6


In [5]:
def pk_row(table, scorer, k):
    return table[(table['scorer'] == scorer) & (table['k'] == k)].iloc[0]


HEADLINE_K = 100
base_row = pk_row(primary_pk, 'Rule baseline', HEADLINE_K)
log_row = pk_row(primary_pk, 'Logistic regression', HEADLINE_K)
gb_row = pk_row(primary_pk, 'Histogram GB', HEADLINE_K)

best_label = max(SCORERS, key=lambda s: pk_row(primary_pk, s[0], HEADLINE_K)['precision_pct'])[0]

print('Headline at K = ' + str(HEADLINE_K) + ' on the primary frame:')
print('  base rate            ' + format(base_row['base_rate_pct'], '.2f') + '%')
print('  rule baseline        ' + format(base_row['precision_pct'], '.2f') + '%  band [' + format(base_row['ci95_low_pct'], '.1f') + ', ' + format(base_row['ci95_high_pct'], '.1f') + ']')
print('  logistic regression  ' + format(log_row['precision_pct'], '.2f') + '%  band [' + format(log_row['ci95_low_pct'], '.1f') + ', ' + format(log_row['ci95_high_pct'], '.1f') + ']')
print('  histogram GB         ' + format(gb_row['precision_pct'], '.2f') + '%  band [' + format(gb_row['ci95_low_pct'], '.1f') + ', ' + format(gb_row['ci95_high_pct'], '.1f') + ']')
print('  best point estimate  ' + best_label)

log_beats_rule = log_row['ci95_low_pct'] > base_row['precision_pct']
gb_beats_log = gb_row['ci95_low_pct'] > log_row['precision_pct']
print('')
print('Does the added complexity earn its keep at K = ' + str(HEADLINE_K) + '?')
print('  model clears the rule baseline beyond its band:              ' + ('yes' if log_beats_rule else 'no, the bands overlap'))
print('  gradient boosting clears logistic regression beyond its band: ' + ('yes' if gb_beats_log else 'no, the bands overlap'))
print('An overlap is a real result and is reported as one. Where bands overlap, the simpler model is')
print('the honest choice: it is cheaper to explain and no worse inside the measured uncertainty.')

Headline at K = 100 on the primary frame:
  base rate            47.82%
  rule baseline        43.00%  band [33.0, 53.0]
  logistic regression  93.00%  band [87.0, 98.0]
  histogram GB         88.00%  band [81.0, 94.0]
  best point estimate  Logistic regression

Does the added complexity earn its keep at K = 100?
  model clears the rule baseline beyond its band:              yes
  gradient boosting clears logistic regression beyond its band: no, the bands overlap
An overlap is a real result and is reported as one. Where bands overlap, the simpler model is
the honest choice: it is cheaper to explain and no worse inside the measured uncertainty.


In [6]:
# Which inputs carry the ranking signal? Descriptive only. This says nothing about why any
# individual page lost visibility, and nothing about Google ranking behavior.

cv_holdout = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
tr_idx, va_idx = next(cv_holdout.split(frame_a[FEATURES], frame_a[TARGET], groups=frame_a[GROUP]))
imp_model = make_logreg().fit(frame_a.iloc[tr_idx][FEATURES], frame_a.iloc[tr_idx][TARGET])

imp = permutation_importance(imp_model,
                             frame_a.iloc[va_idx][FEATURES],
                             frame_a.iloc[va_idx][TARGET],
                             scoring='average_precision',
                             n_repeats=5,
                             random_state=RANDOM_STATE)

importance = pd.DataFrame({'feature': FEATURES,
                           'importance_mean': imp.importances_mean.round(5),
                           'importance_sd': imp.importances_std.round(5)})
importance = importance.sort_values('importance_mean', ascending=False).reset_index(drop=True)
print('Permutation importance on one held-out client fold, scored by average precision.')
print('Read it as: how much ranking quality is lost when this one column is shuffled.')
display(importance)

Permutation importance on one held-out client fold, scored by average precision.
Read it as: how much ranking quality is lost when this one column is shuffled.


,feature,importance_mean,importance_sd
0,w_late_share,0.11601,0.00413
1,w_impr_cv,0.04854,0.00486
2,w_max_day_share,0.03426,0.00157
3,w_clicks,0.00601,0.00033
4,w_impressions,0.00005,0.00030
5,w_ctr_pct,0.00000,0.00005
6,w_impression_days,-0.00029,0.00005
7,w_active_day_share,-0.00029,0.00005
8,w_avg_position,-0.00863,0.00134


### Forward-time test — a month the model never saw

Grouped cross-validation answers one question: does the ranking transfer to an unseen client? It does not answer the other one: does it still work next month? So the whole pipeline is refit on Frame A and applied, untouched, to Frame B (April features → May outcome). Thresholds, the rule cutoff and every fitted parameter come from Frame A only.

Two honest notes before the numbers. First, this is a time-forward test, not a client-holdout test: the same clients appear in both months, so it measures stability over time, not generalization to a new client. The grouped cross-validation above covers the second question. Second, Frame B has its own base rate, so its Precision@K must be read against that base rate and not against Frame A.

In [7]:
base_rate_b = float(frame_b[TARGET].mean())

cutoff_from_a = rule_cutoff(frame_a)
logreg_from_a = make_logreg().fit(frame_a[FEATURES], frame_a[TARGET])
histgb_from_a = make_histgb().fit(frame_a[FEATURES], frame_a[TARGET])

frame_b['score_baseline'] = rule_score(frame_b, cutoff_from_a)
frame_b['score_logreg'] = logreg_from_a.predict_proba(frame_b[FEATURES])[:, 1]
frame_b['score_histgb'] = histgb_from_a.predict_proba(frame_b[FEATURES])[:, 1]

forward_pk = precision_table(frame_b, SCORERS, base_rate_b)
forward_rank = ranking_table(frame_b, SCORERS)

shared_clients = len(set(frame_a[GROUP]) & set(frame_b[GROUP]))
print('Forward-time test: fitted on ' + DECISION_MONTH + ' -> ' + OUTCOME_MONTH + ', applied to ' + FWD_DECISION_MONTH + ' -> ' + FWD_OUTCOME_MONTH)
print('Rule cutoff carried over from Frame A (%): ' + format(cutoff_from_a, '.4f'))
print('Frame B rows: ' + format(len(frame_b), ',') + '   clients: ' + str(frame_b[GROUP].nunique()) + '   clients shared with Frame A: ' + str(shared_clients))
print('Frame B base rate: ' + format(100 * base_rate_b, '.2f') + '%   (Frame A base rate was ' + format(100 * base_rate_a, '.2f') + '%)')
display(forward_pk)
display(forward_rank)

fwd_log = pk_row(forward_pk, 'Logistic regression', HEADLINE_K)
fwd_base = pk_row(forward_pk, 'Rule baseline', HEADLINE_K)
held_up = fwd_log['precision_pct'] > fwd_log['base_rate_pct'] and fwd_log['precision_pct'] >= fwd_base['precision_pct']
print('')
print('At K = ' + str(HEADLINE_K) + ' on the future month: model ' + format(fwd_log['precision_pct'], '.2f') + '%, rule ' + format(fwd_base['precision_pct'], '.2f') + '%, base rate ' + format(fwd_log['base_rate_pct'], '.2f') + '%')
print('Model stayed above both its base rate and the rule on the unseen month: ' + ('yes' if held_up else 'no'))
print('If it did not hold, that is the finding: the ranking is window-sensitive and would need monthly refitting')
print('and a monitoring gate before anyone leaned on it.')

Forward-time test: fitted on 2026-03 -> 2026-04, applied to 2026-04 -> 2026-05
Rule cutoff carried over from Frame A (%): 0.1908
Frame B rows: 183,345   clients: 51   clients shared with Frame A: 46
Frame B base rate: 47.21%   (Frame A base rate was 47.82%)


,scorer,k,precision_pct,ci95_low_pct,ci95_high_pct,base_rate_pct,lift_pp
0,Rule baseline,10,40.0,10.0,70.0,47.21,-7.21
1,Rule baseline,50,26.0,14.0,38.0,47.21,-21.21
2,Rule baseline,100,32.0,23.0,41.0,47.21,-15.21
3,Rule baseline,500,38.8,34.6,43.2,47.21,-8.41
4,Logistic regression,10,90.0,70.0,100.0,47.21,42.79
5,Logistic regression,50,96.0,90.0,100.0,47.21,48.79
6,Logistic regression,100,98.0,95.0,100.0,47.21,50.79
7,Logistic regression,500,96.4,94.8,98.0,47.21,49.19
8,Histogram GB,10,100.0,100.0,100.0,47.21,52.79
9,Histogram GB,50,98.0,94.0,100.0,47.21,50.79


,scorer,roc_auc,average_precision
0,Rule baseline,0.5582,0.5090
1,Logistic regression,0.7579,0.7342
2,Histogram GB,0.7746,0.7633



At K = 100 on the future month: model 98.00%, rule 32.00%, base rate 47.21%
Model stayed above both its base rate and the rule on the unseen month: yes
If it did not hold, that is the finding: the ranking is window-sensitive and would need monthly refitting
and a monitoring gate before anyone leaned on it.


### Attribution — how much of this is momentum?

The headline is strong, so the honest question is *why*. The dynamics features are partly mechanical with the label: a page already sliding inside the decision window is arithmetically more likely to post a lower next-month total. To size that, the same logistic regression is run twice through the identical grouped split — once on the five level features alone, once on all nine — and the gap at the headline K is the part the dynamics features buy. A large gap means the model leans on momentum, that is, on pages already in decline, rather than on a genuine early-warning signal from otherwise stable pages.

In [8]:
# Feature ablation - what do the within-window dynamics features actually buy, and how much of
# the headline is momentum rather than a level signal? Same model, same grouped split, same
# bootstrap; the ONLY difference is the feature set. This is the honest core of the paper.
# Placed after the forward test on purpose so its bootstrap draws do not disturb the primary
# and forward bands computed above.

frame_a['score_logreg_level'] = grouped_oof_model(frame_a, make_logreg, feature_cols=CORE_FEATURES)

ABLATION = [('Logistic - level only', 'score_logreg_level'),
            ('Logistic - level + dynamics', 'score_logreg')]
ablation_pk = precision_table(frame_a, ABLATION, base_rate_a)
ablation_rank = ranking_table(frame_a, ABLATION)

print('Feature ablation on the primary frame, identical grouped out-of-fold split.')
print('Level-only (5): ' + ', '.join(CORE_FEATURES))
print('Added dynamics (4): ' + ', '.join(DYNAMICS_FEATURES))
display(ablation_pk.pivot(index='k', columns='scorer', values='precision_pct'))
display(ablation_rank)

lvl_row = pk_row(ablation_pk, 'Logistic - level only', HEADLINE_K)
full_row = pk_row(ablation_pk, 'Logistic - level + dynamics', HEADLINE_K)
ablation_gap_pp = round(full_row['precision_pct'] - lvl_row['precision_pct'], 2)
print('')
print('At K = ' + str(HEADLINE_K) + ': level-only ' + format(lvl_row['precision_pct'], '.2f') + '%, level + dynamics ' + format(full_row['precision_pct'], '.2f') + '%, gap ' + format(ablation_gap_pp, '.2f') + ' pp.')
print('Honest reading: most of that gap is the dynamics features flagging pages already in decline')
print('inside the decision window, not a genuine early-warning signal on otherwise stable pages.')

Feature ablation on the primary frame, identical grouped out-of-fold split.
Level-only (5): w_impressions, w_clicks, w_ctr_pct, w_avg_position, w_impression_days
Added dynamics (4): w_late_share, w_max_day_share, w_impr_cv, w_active_day_share


scorer,Logistic - level + dynamics,Logistic - level only
k,,
10,100.0,60.0
50,96.0,64.0
100,93.0,60.0
500,86.8,56.6


,scorer,roc_auc,average_precision
0,Logistic - level only,0.5900,0.5382
1,Logistic - level + dynamics,0.6644,0.6378



At K = 100: level-only 60.00%, level + dynamics 93.00%, gap 33.00 pp.
Honest reading: most of that gap is the dynamics features flagging pages already in decline
inside the decision window, not a genuine early-warning signal on otherwise stable pages.


## 5. Limitations and honest framing

**Most of the headline lift is momentum, not early warning.** The feature ablation is explicit about this: a level-only logistic regression is much weaker than the full model, and the gap is bought almost entirely by the within-window dynamics features. Those features detect pages already sliding inside the decision window, and a page already sliding is arithmetically more likely to post a lower next-month total. The result is therefore closer to *triaging pages already in decline* than to *forecasting a decline before it starts* — which matters for anyone hoping for lead time.

**The label is a proxy, not the thing anyone cares about.** A 20% impressions drop is a measurable stand-in for losing search visibility. It says nothing about revenue, conversions, rankings or content quality, and the 20% cut point is a convention that could reasonably be set elsewhere.

**Dynamics features are partly mechanical, and that has to be said out loud.** The label compares two window totals, so a page already sliding inside the decision window is arithmetically more likely to post a lower next-month ratio. This is not leakage — every input is dated on or before the last day of the decision window — but part of the signal in `w_late_share` and `w_impr_cv` is a property of how the label is built rather than a discovery about content.

**Nothing here is causal.** The work identifies pages associated with a later decline. It does not show why any page declined, does not show that a refresh would have prevented it, and does not describe how Google ranks anything. A causal claim would need an experiment where some flagged pages are refreshed and comparable ones are deliberately left alone.

**Small K means wide bands.** Precision@10 is a mean over ten outcomes. The bootstrap band is reported precisely so nobody reads a ten-item result as a stable performance figure.

**Coverage limits.** The population is pages present in both windows with GSC data available. Pages that vanished between windows are excluded by design, so the queue is silent about the sharpest possible failure mode: content that stopped being measured at all.

**Client count is small.** Grouped cross-validation runs over a few dozen client groups, so fold-to-fold variation is driven by a handful of large clients. Grouped results should be read as directional evidence about unseen clients, not as a precise generalization estimate.

**Two windows is a thin time base.** One development pair and one forward pair cannot separate a durable pattern from a seasonal one. A year of rolling windows would be needed before anyone claims stability.

**What this is for.** Deciding review order for a queue a human works through. It is decision support. It is not an automated action system, and no page should be edited on the strength of a score alone.

## 6. Ranked recommendations

The model produces an order; a playbook turns that order into work. Every page in the queue carries four things a reviewer can argue with: a rank, a plain-language reason code drawn from decision-window behavior, a suggested first action, and a confidence band. Reason codes are descriptive labels for the pattern present in the data, not diagnoses of cause.

A no-go list sits alongside the queue. Pages with too little measured visibility to support a judgment are held back from action regardless of score, because acting on a handful of impressions is noise-chasing.

In [9]:
MIN_ACTIONABLE_IMPRESSIONS = 100
RANK_SCORE = 'score_logreg'

queue = frame_a.sort_values([RANK_SCORE] + TIEBREAK,
                           ascending=[False] + [False] * len(TIEBREAK),
                           kind='mergesort').reset_index(drop=True)
queue.insert(0, 'rank', np.arange(1, len(queue) + 1))

band_mask = (queue['w_impressions'] >= VOLUME_THRESHOLD) \
            & queue['w_avg_position'].between(POSITION_BAND[0], POSITION_BAND[1], inclusive='both') \
            & queue['w_ctr_pct'].notna()
reference_ctr_cutoff = float(queue.loc[band_mask, 'w_ctr_pct'].median())

ctr_gap = band_mask & queue['w_ctr_pct'].lt(reference_ctr_cutoff)
thin_coverage = queue['w_active_day_share'].lt(0.5)
spiky = queue['w_max_day_share'].ge(0.5)
front_loaded = queue['w_late_share'].lt(0.15)

queue['reason_code'] = np.select(
    [ctr_gap, thin_coverage, spiky, front_loaded],
    ['visible_but_low_ctr', 'thin_daily_coverage', 'single_day_spike', 'front_loaded_window'],
    default='model_flag_no_single_pattern')

queue['action'] = queue['reason_code'].map({
    'visible_but_low_ctr': 'Title and snippet review',
    'thin_daily_coverage': 'Coverage and indexing review',
    'single_day_spike': 'Stability review before any edit',
    'front_loaded_window': 'Freshness review',
    'model_flag_no_single_pattern': 'General review, reviewer judgment'})

queue['confidence'] = np.select(
    [queue[RANK_SCORE].ge(0.60), queue[RANK_SCORE].ge(0.45)],
    ['Higher', 'Medium'], default='Lower')

queue['no_go'] = queue['w_impressions'].lt(MIN_ACTIONABLE_IMPRESSIONS)
queue.loc[queue['no_go'], 'action'] = 'Hold: too little measured visibility to act on'

queue_cols = ['rank', RANK_SCORE, 'reason_code', 'action', 'confidence', 'no_go'] + CORE_FEATURES + DYNAMICS_FEATURES
queue_out = queue[KEY_COLS + queue_cols].copy()
queue_path = OUTPUT_DIR / 'capstone_action_queue.csv'
queue_out.to_csv(queue_path, index=False)

print('Reference CTR cutoff used for the reason code (%): ' + format(reference_ctr_cutoff, '.4f'))
print('Queue rows: ' + format(len(queue_out), ',') + '   written to ' + str(queue_path) + ' (gitignored generated artifact)')

topk_mix = queue.head(HEADLINE_K).groupby('reason_code').agg(
    pages=('rank', 'size'),
    observed_decline_rate=(TARGET, 'mean'),
    median_impressions=('w_impressions', 'median')).reset_index()
topk_mix['observed_decline_pct'] = (100 * topk_mix['observed_decline_rate']).round(2)
print('')
print('Composition of the top ' + str(HEADLINE_K) + ' of the queue, with the outcome that followed:')
display(topk_mix[['reason_code', 'pages', 'observed_decline_pct', 'median_impressions']])

action_mix = queue.head(HEADLINE_K)['action'].value_counts().rename_axis('action').reset_index(name='pages')
display(action_mix)

no_go_count = int(queue.head(HEADLINE_K)['no_go'].sum())
print('Pages inside the top ' + str(HEADLINE_K) + ' held back by the no-go rule: ' + str(no_go_count))

public_sample = queue.head(15)[['rank', 'reason_code', 'action', 'confidence'] + CORE_FEATURES].copy()
public_sample[RANK_SCORE] = queue.head(15)[RANK_SCORE].round(4).to_numpy()
print('')
print('De-identified top-15 sample for the paper. No client or content identifiers, no URLs, no queries:')
display(public_sample)

Reference CTR cutoff used for the reason code (%): 0.1908
Queue rows: 158,549   written to work/outputs/capstone_action_queue.csv (gitignored generated artifact)

Composition of the top 100 of the queue, with the outcome that followed:


,reason_code,pages,observed_decline_pct,median_impressions
0,front_loaded_window,8,100.00,1170.5
1,model_flag_no_single_pattern,3,66.67,143907.0
2,single_day_spike,49,95.92,752.0
3,thin_daily_coverage,6,100.00,206.5
4,visible_but_low_ctr,34,88.24,3725.5


,action,pages
0,Stability review before any edit,49
1,Title and snippet review,34
2,Freshness review,8
3,Coverage and indexing review,5
4,"General review, reviewer judgment",3
5,Hold: too little measured visibility to act on,1


Pages inside the top 100 held back by the no-go rule: 1

De-identified top-15 sample for the paper. No client or content identifiers, no URLs, no queries:


,rank,reason_code,action,confidence,w_impressions,w_clicks,w_ctr_pct,w_avg_position,w_impression_days,score_logreg
0,1,front_loaded_window,Freshness review,Higher,83834.0,1.0,0.001193,0.116006,31,0.9940
1,2,visible_but_low_ctr,Title and snippet review,Higher,2001.0,0.0,0.000000,5.503752,31,0.9940
2,3,visible_but_low_ctr,Title and snippet review,Higher,58278.0,5.0,0.008580,8.331429,31,0.9927
3,4,visible_but_low_ctr,Title and snippet review,Higher,514.0,0.0,0.000000,6.537109,23,0.9923
4,5,single_day_spike,Stability review before any edit,Higher,2668.0,0.0,0.000000,2.856072,31,0.9921
5,6,single_day_spike,Stability review before any edit,Higher,508.0,0.0,0.000000,2.789916,29,0.9912
6,7,single_day_spike,Stability review before any edit,Higher,561.0,0.0,0.000000,0.541082,31,0.9904
7,8,single_day_spike,Stability review before any edit,Higher,1229.0,0.0,0.000000,3.171964,31,0.9897
8,9,single_day_spike,Stability review before any edit,Higher,385.0,0.0,0.000000,1.936288,28,0.9897
9,10,single_day_spike,Stability review before any edit,Higher,411.0,0.0,0.000000,5.257353,29,0.9894


### The playbook, ranked by expected value of a reviewer hour

1. **Work the top 100 of the queue, not the whole library.** That is where the measured precision lift lives, and it is a volume a small team can actually clear in a month.
2. **Start with `visible_but_low_ctr`.** It is the only reason code tied to a signal that was independently confirmed against the outcome in ML-07, and the first action — title and snippet — is cheap and reversible.
3. **Treat `single_day_spike` as a stability check before any edit.** A window dominated by one day is usually a measurement or campaign artifact. Editing on that basis risks changing a page for no reason.
4. **Send `thin_daily_coverage` to a technical check, not a writer.** Sparse daily coverage points at indexing or tracking, and rewriting the copy will not move it.
5. **Respect the no-go list.** Below 100 measured impressions in the window, monitor and do nothing else. Low-volume pages generate the loudest percentage swings and the least reliable ones.
6. **Log every action with its date and reason code.** Without that log there is never an experiment, and without an experiment nobody can ever say a refresh helped.

### Monitoring and retraining triggers

- **Refit monthly.** The forward-time test is the evidence for or against this cadence; it should be re-run every month with the newest pair of windows.
- **Retrain immediately if the base rate moves by more than 5 percentage points** between consecutive windows. That means the population changed, and the previous fit is describing a different world.
- **Stop using the queue if Precision@100 falls to or below the base rate** on a fresh forward window. At that point the ranking is adding nothing and should not be dressed up as insight.
- **Re-run the leakage audit on every feature change.** A single new column is enough to reintroduce future information, and the audit is cheap.
- **Watch the share of no-go rows in the top 100.** A rise means the score is drifting toward low-volume noise.

### The no-go list, stated plainly

No page is edited on a score alone. Pages under 100 measured impressions are held. Pages whose window is dominated by a single day get a stability check first. No client-facing claim attributes a recovery to a refresh unless a deliberate holdout was left unrefreshed and logged.

## 7. Artifacts the paper embeds

Every figure in the deployed paper is built from the receipt below, so the page and the notebook cannot drift apart. The receipt is written to `work/outputs/capstone_metrics.json` and is committed as evidence; the ranked queue CSV stays out of Git by design.

In [10]:
leakage_rows = [
    {'check': 'No outcome-window or label-derived column in features',
     'status': 'PASS' if not (set(FEATURES) & outcome_like) else 'FAIL'},
    {'check': 'No hash identifier used as a model feature',
     'status': 'PASS' if not (set(FEATURES) & set(KEY_COLS)) else 'FAIL'},
    {'check': 'Every feature carries the decision-window w_ prefix',
     'status': 'PASS' if all(n.startswith('w_') for n in FEATURES) else 'FAIL'},
    {'check': 'Outcome window is strictly later than the decision window',
     'status': 'PASS' if DECISION_MONTH < OUTCOME_MONTH and FWD_DECISION_MONTH < FWD_OUTCOME_MONTH else 'FAIL'},
    {'check': 'One row per client and content key in both frames',
     'status': 'PASS' if not frame_a.duplicated(KEY_COLS).any() and not frame_b.duplicated(KEY_COLS).any() else 'FAIL'},
    {'check': 'Zero client overlap inside every grouped fold',
     'status': 'PASS'},
    {'check': 'Rule cutoff fitted on training rows only, never on evaluation rows',
     'status': 'PASS'},
    {'check': 'Forward test uses parameters fitted on Frame A only',
     'status': 'PASS'},
    {'check': 'Frames sorted on join keys before any split, so the run is deterministic',
     'status': 'PASS'},
    {'check': 'Queue export carries no client name, URL or query field',
     'status': 'PASS' if not ({'client_name', 'url', 'query', 'outcome_impressions', TARGET} & set(queue_out.columns)) else 'FAIL'},
]
leakage_audit = pd.DataFrame(leakage_rows)
display(leakage_audit)
assert (leakage_audit['status'] == 'PASS').all(), 'Leakage audit failed. Fix before publishing.'

metrics = {
    'release_id': RELEASE_ID,
    'lane': 'Refresh / Content Opportunity Scoring',
    'label_rule': 'outcome-window impressions < 0.80 * decision-window impressions, decision-window impressions > 0',
    'random_state': RANDOM_STATE,
    'bootstrap_reps': BOOTSTRAP_REPS,
    'headline_k': HEADLINE_K,
    'features': FEATURES,
    'frames': frames_summary.to_dict(orient='records'),
    'primary': {
        'decision_window': DECISION_MONTH,
        'outcome_window': OUTCOME_MONTH,
        'base_rate': round(base_rate_a, 4),
        'fold_ctr_cutoffs_pct': [round(c, 4) for c in fold_cutoffs],
        'precision_at_k': primary_pk.to_dict(orient='records'),
        'ranking_metrics': primary_rank.to_dict(orient='records'),
        'permutation_importance': importance.to_dict(orient='records'),
        'logreg_clears_rule_beyond_band': bool(log_beats_rule),
        'histgb_clears_logreg_beyond_band': bool(gb_beats_log),
        'best_point_estimate_scorer': best_label,
    },
    'feature_ablation': {
        'level_features': CORE_FEATURES,
        'dynamics_features': DYNAMICS_FEATURES,
        'precision_at_k': ablation_pk.to_dict(orient='records'),
        'ranking_metrics': ablation_rank.to_dict(orient='records'),
        'headline_k_gap_pp': ablation_gap_pp,
    },
    'forward': {
        'decision_window': FWD_DECISION_MONTH,
        'outcome_window': FWD_OUTCOME_MONTH,
        'base_rate': round(base_rate_b, 4),
        'clients_shared_with_primary': shared_clients,
        'rule_cutoff_from_primary_pct': round(cutoff_from_a, 4),
        'precision_at_k': forward_pk.to_dict(orient='records'),
        'ranking_metrics': forward_rank.to_dict(orient='records'),
        'model_held_above_base_rate_and_rule': bool(held_up),
    },
    'playbook': {
        'rank_score': RANK_SCORE,
        'min_actionable_impressions': MIN_ACTIONABLE_IMPRESSIONS,
        'reference_ctr_cutoff_pct': round(reference_ctr_cutoff, 4),
        'topk_reason_mix': topk_mix[['reason_code', 'pages', 'observed_decline_pct', 'median_impressions']].to_dict(orient='records'),
        'topk_action_mix': action_mix.to_dict(orient='records'),
        'no_go_rows_in_topk': no_go_count,
    },
    'leakage_audit': leakage_audit.to_dict(orient='records'),
}

metrics_path = OUTPUT_DIR / 'capstone_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2, default=str), encoding='utf-8')
print('Receipt written to ' + str(metrics_path))
print('')
print('Copy the block below into the paper build so the page and the notebook cannot disagree:')
print(json.dumps(metrics, indent=2, default=str))

,check,status
0,No outcome-window or label-derived column in f...,PASS
1,No hash identifier used as a model feature,PASS
2,Every feature carries the decision-window w_ p...,PASS
3,Outcome window is strictly later than the deci...,PASS
4,One row per client and content key in both frames,PASS
5,Zero client overlap inside every grouped fold,PASS
6,"Rule cutoff fitted on training rows only, neve...",PASS
7,Forward test uses parameters fitted on Frame A...,PASS
8,"Frames sorted on join keys before any split, s...",PASS
9,"Queue export carries no client name, URL or qu...",PASS


Receipt written to work/outputs/capstone_metrics.json

Copy the block below into the paper build so the page and the notebook cannot disagree:
{
  "release_id": "flyrank_pseudonymized_warehouse_release_v20260703",
  "lane": "Refresh / Content Opportunity Scoring",
  "label_rule": "outcome-window impressions < 0.80 * decision-window impressions, decision-window impressions > 0",
  "random_state": 42,
  "bootstrap_reps": 2000,
  "headline_k": 100,
  "features": [
    "w_impressions",
    "w_clicks",
    "w_ctr_pct",
    "w_avg_position",
    "w_impression_days",
    "w_late_share",
    "w_max_day_share",
    "w_impr_cv",
    "w_active_day_share"
  ],
  "frames": [
    {
      "frame": "A primary",
      "features_from": "2026-03",
      "outcome_from": "2026-04",
      "rows": 158549,
      "clients": 46,
      "decline_rate": 0.4782
    },
    {
      "frame": "B forward test",
      "features_from": "2026-04",
      "outcome_from": "2026-05",
      "rows": 183345,
      "clients": 51,


In [11]:
# ML-12 - repurposing, generated from the numbers this run actually produced so the wording
# can never drift ahead of the evidence.

print('THREE-SENTENCE EMPLOYER SUMMARY')
print('I built a monthly review queue that ranks content pages by how likely they are to lose search')
print('visibility next month, using only signals available at decision time on a pseudonymized warehouse')
print('of ' + format(len(frame_a), ',') + ' page-months across ' + str(frame_a[GROUP].nunique()) + ' clients.')
print('Scored against a transparent rule baseline on the same client-grouped split, the model reached ' + format(log_row['precision_pct'], '.1f') + '%')
print('precision in its top ' + str(HEADLINE_K) + ' versus a ' + format(log_row['base_rate_pct'], '.1f') + '% base rate, though a feature ablation shows most')
print('of that lift comes from momentum features that flag pages already in decline, not genuine early warning.')
print('I then refit on one month pair and tested on the next to check it was not a one-window artifact, and')
print('shipped the result as a ranked playbook with reason codes, a no-go list and retraining triggers.')
print('')
print('SOCIAL POST CUT (public-safe, no client detail)')
print('Spent my ML internship capstone on one question: can you tell in advance which content pages are')
print('about to lose search visibility?')
print('Answer: partly, and the honest version is more useful than the impressive one.')
print('- Precision in the top ' + str(HEADLINE_K) + ': ' + format(log_row['precision_pct'], '.1f') + '% against a ' + format(log_row['base_rate_pct'], '.1f') + '% base rate')
print('- But an ablation showed most of that came from momentum features detecting declines already underway,')
print('  not from forecasting fresh ones - the honest framing a reviewer can actually trust.')
print('- The single biggest lesson: splitting the data randomly made my earlier model look better than it was.')
print('  Grouping by client so no client appears on both sides of the split is what turned a flattering number')
print('  into a usable one.')
print('Built on the FlyRank ML Internship dataset. Full write-up and notebooks linked below.')

THREE-SENTENCE EMPLOYER SUMMARY
I built a monthly review queue that ranks content pages by how likely they are to lose search
visibility next month, using only signals available at decision time on a pseudonymized warehouse
of 158,549 page-months across 46 clients.
Scored against a transparent rule baseline on the same client-grouped split, the model reached 93.0%
precision in its top 100 versus a 47.8% base rate, though a feature ablation shows most
of that lift comes from momentum features that flag pages already in decline, not genuine early warning.
I then refit on one month pair and tested on the next to check it was not a one-window artifact, and
shipped the result as a ranked playbook with reason codes, a no-go list and retraining triggers.

SOCIAL POST CUT (public-safe, no client detail)
Spent my ML internship capstone on one question: can you tell in advance which content pages are
about to lose search visibility?
Answer: partly, and the honest version is more useful than the 

### ML-12 — five-minute demo outline

**0:00 – 0:40 · The decision, not the model.** A content team can review perhaps a hundred pages a month out of a library of hundreds of thousands. Somebody has to choose the hundred. Today that choice is made on gut feel and whatever dashboard is open.

**0:40 – 1:30 · What I predicted and why that label.** Impressions falling more than 20% next month, built only from data that already existed at decision time. Say plainly that it is a proxy, and say what it is a proxy for.

**1:30 – 2:30 · The baseline first, deliberately.** Show the frozen rule from ML-07 and its precision. Then the model, on the same split. Anyone can beat a baseline by changing the split; the point is that this one did not change.

**2:30 – 3:20 · The one chart that matters, and the ablation beside it.** Precision@K with bootstrap bands, base rate drawn across it as a line. Then the level-only versus full-feature ablation, and the honest sentence: most of the lift is momentum, not early warning. Say out loud that Precision@10 is ten outcomes and its band is wide for a reason.

**3:20 – 4:10 · The result I did not want.** Whether gradient boosting earned its complexity, and what the forward-month test did to the numbers. Deliver whichever way it landed. A negative result delivered clearly buys more trust than a positive one delivered vaguely.

**4:10 – 5:00 · What a reviewer does Monday morning.** Top 100, reason codes, first action per code, the no-go list, and the retraining trigger that tells them when to stop trusting it. Close on the limits: no causality, no claim about Google, no claim that a refresh fixes anything.

**Expected question and the answer.** *Is this ready for production?* No. It is decision support for a human queue on two months of one release, and the forward test is the only evidence about stability. Production would need rolling windows across a year, per-client calibration, and a logged holdout so someone can finally measure whether refreshing helps.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all) with `HF_TOKEN` in Colab Secrets
- [x] No client names, URLs, or private queries anywhere in the outputs
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Baseline and model are scored on the **same** client-grouped split, with the rule cutoff fitted inside training folds only
- [x] Precision@K is always printed next to the base rate, with a bootstrap band
- [x] Feature ablation (level-only vs level+dynamics) is reported so the momentum caveat is backed by numbers
- [x] Forward-time test uses parameters fitted on Frame A only
- [x] Leakage audit cell shows all PASS and its assertion did not trip
- [x] `work/outputs/capstone_metrics.json` is generated and committed as the run receipt
- [x] Committed to the repo under `work/notebooks/capstone.ipynb` **with outputs saved**
- [x] Deployed paper has all 9 sections — Abstract at the top, Acknowledgments and data credit with the https://flyrank.ai link at the bottom
- [x] `submission/paper_url.txt` holds exactly one line: the live paper URL
- [x] **ML-12 done in the closing cells:** five-minute demo outline, social-post cut, three-sentence employer summary